# Flow from differential pressure: the GUI backend in a notebook

The network the GUI draws is also a Python object. This notebook loads the
shipped GUI example *ISO 5167-2 orifice metering run*, drives it from a small
`pandas` table of pressures, and gets the mass flow back for every row, with no
web server running.

The network: air at 300 K, a 100 mm pipe (20 D), a beta = 0.5 square-edged
plate with flange taps, and a 10 D pipe, between a supply and a sink, both
total-pressure boundaries. Each row of the table sets the supply pressure and
the overall pressure difference; the solver finds the flow that the pipes and
the plate pass at that difference.

The same network opens in the GUI from **Examples -> ISO 5167-2 orifice
metering run**. `python/examples/papers/iso5167_orifice_metering.py` checks it
against the standard's own equations.

## Load the network

`NetworkRunner` is what the GUI's server calls. It reads a network saved from
the GUI (here the example that ships with `combaero-gui`). The first cell finds
the repository's `gui/` package when this runs from a checkout, and the
installed `combaero-gui` otherwise.

In [ ]:
import sys
from pathlib import Path

for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "gui" / "backend" / "runner.py").is_file():
        sys.path.insert(0, str(parent))
        break

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import gui
from gui.backend.runner import NetworkRunner

network_file = Path(gui.__file__).parent / "examples" / "iso5167_orifice_metering.json"
runner = NetworkRunner.from_file(network_file)

design = runner.solve()
print(f"success: {design.success}")
print(f"m_dot at the saved boundaries: {design.get('plate.m_dot'):.4f} kg/s")

## The cases

One row per operating point. Columns are overrides, written
`<node>.<field>`: the node's label or id, then the field the GUI shows for it.
Here `supply.Pt` and `sink.Pt` are the two boundaries' total pressures [Pa].
Three supply pressures, four differences each. The index names the rows; it
carries through to the results.

In [ ]:
supply_bar = [1.5, 3.0, 6.0]
dp_kPa = [1.0, 2.5, 5.0, 10.0]

rows = [
    {"P_supply [bar]": p, "dp [kPa]": dp, "supply.Pt": p * 1e5, "sink.Pt": p * 1e5 - dp * 1e3}
    for p in supply_bar
    for dp in dp_kPa
]
cases = pd.DataFrame(rows)
cases.index = [f"{r['P_supply [bar]']:g} bar, {r['dp [kPa]']:g} kPa" for r in rows]
cases

## Solve every row

`sweep` takes only the override columns, solves each row in turn (each solve
starts from the previous one's answer), and returns one row per case. `metrics`
picks the results to keep; leave it out to keep everything. Results come back
under `outputs`, indexed `(entity, quantity)`, with the solve status under
`_solve`. Check `success` before trusting a number.

In [ ]:
metrics = [
    "plate.m_dot",
    "plate.dP_tap",
    "plate.dP_permanent",
    "plate.Cd",
    "plate.expansibility",
    "plate.Re",
    "supply.rho",
]
results = runner.sweep(cases[["supply.Pt", "sink.Pt"]], metrics=metrics)

out = results["outputs"]
assert out["_solve", "success"].all(), out["_solve"]
out["_solve"]

## A results table

The `outputs` columns are named by the entity's label. Flatten the ones we want
into a plain table next to the inputs, using the units the results carry in
`results.attrs["units"]`.

In [ ]:
plate = out.xs("Orifice plate, beta 0.5", axis=1, level="entity")

table = cases[["P_supply [bar]", "dp [kPa]"]].copy()
table["m_dot [kg/s]"] = plate["m_dot"]
table["dP_tap [kPa]"] = plate["dP_tap"] / 1e3
table["dP_permanent [kPa]"] = plate["dP_permanent"] / 1e3
table["Cd [-]"] = plate["Cd"]
table["expansibility [-]"] = plate["expansibility"]
table["Re [-]"] = plate["Re"]
table.round(4)

## Plot

Flow against the difference, one line per supply pressure. The second panel
divides by `sqrt(rho * dp)`, with `rho` the supply density: the lines nearly
collapse, which is the square-root law every orifice meter rests on. The
spread that is left, a few per cent, grows with dp / P: that is the
expansibility, the gas expanding through the bore. The small rest is the
discharge coefficient's Reynolds number dependence.

In [ ]:
rho = out["Supply", "rho"]
flow_coefficient = table["m_dot [kg/s]"] / (rho * table["dp [kPa]"] * 1e3) ** 0.5

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
for p, group in table.groupby("P_supply [bar]"):
    ax1.plot(group["dp [kPa]"], group["m_dot [kg/s]"], "o-", label=f"{p:g} bar")
    ax2.plot(group["dp [kPa]"], flow_coefficient[group.index], "o-", label=f"{p:g} bar")
ax1.set(xlabel="overall dp [kPa]", ylabel="m_dot [kg/s]", title="Mass flow")
ax2.set(
    xlabel="overall dp [kPa]",
    ylabel="m_dot / sqrt(rho dp) [m^2]",
    title="Flow coefficient",
)
ax1.legend(title="supply")
ax2.legend(title="supply")
fig.tight_layout()
plt.show()

## One case in full

Any single case can be solved on its own for the full detail: every node and
element, every quantity, the same table the GUI's CSV export writes.

In [ ]:
one = runner.solve({"supply.Pt": 3.0e5, "sink.Pt": 3.0e5 - 5.0e3})
detail = one.to_dataframe().set_index("label")
detail.loc[
    ["Upstream pipe, 20 D", "Orifice plate, beta 0.5", "Downstream pipe, 10 D"],
    ["m_dot [kg/s]", "P_in [Pa]", "P_out [Pa]", "dPt [Pa]", "mach_in [-]", "Re [-]"],
]

## Where to go next

- Any field the GUI shows can be an override column: `supply.Tt`,
  `plate.diameter`, a mass-flow boundary's `m_dot`.
- Save a network of your own from the GUI and load it the same way with
  `NetworkRunner.from_file`.
- `gui.backend.results.load_results` reads a sweep's table, or a GUI CSV
  export, back into this one-row-per-case form.